### Load Dependencies and Data

In [30]:
# Set dependencies
import pandas as pd
import numpy as np
import torch
from pyprojroot import here
from sentence_transformers import SentenceTransformer


# Project path anchors
REPO_ROOT = here()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
MODEL_DIR = REPO_ROOT / "data" / "models"



In [31]:
# Load Data

def load_split(dataset: str, split: str) -> pd.DataFrame:
    X = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_X_{split}.csv")
    y = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_Y_{split}.csv")
    df = pd.concat([X, y], axis=1)
    df["dataset"] = dataset
    df["split"] = split
    return df

all_data = pd.concat(
    [load_split(ds, sp) for ds in ("multiturn", "singleturn") for sp in ("train", "val", "test")],
    ignore_index=True,
)

all_data["conversation_id"] = all_data["conversation_id"].astype(str)


### Final Check for Balance, Length, and Quality

In [32]:
# Check overall structure
all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8148 entries, 0 to 8147
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   conversation_id  8148 non-null   str  
 1   conversation     8148 non-null   str  
 2   harm             8148 non-null   bool 
 3   dataset          8148 non-null   str  
 4   split            8148 non-null   str  
dtypes: bool(1), str(4)
memory usage: 48.9 MB


In [33]:
# check for leakage across train, val, test
all_data["conversation_id"].duplicated().sum()
all_data.duplicated(subset="conversation").sum()
all_data.groupby("conversation_id")["split"].nunique().gt(1).sum()


np.int64(1014)

In [34]:
# nulls or empty last check
all_data.isna().sum()
(all_data["conversation"].str.strip() == "").sum()

np.int64(0)

In [35]:
# Label balance look
all_data.groupby(["dataset", "split"])["harm"].mean()
all_data.groupby(["dataset", "split"])["harm"].value_counts(normalize=True).unstack()


harm                 False     True 
dataset    split                    
multiturn  test   0.515337  0.484663
           train  0.496318  0.503682
           val    0.495706  0.504294
singleturn test   0.554601  0.445399
           train  0.475041  0.524959
           val    0.465031  0.534969

In [36]:
# length and structure by source

all_data["n_chars"] = all_data["conversation"].str.len()
all_data["n_turns"] = all_data["conversation"].str.count("USER:|ASSISTANT:")
all_data.groupby("dataset")[["n_chars", "n_turns"]].describe()



n_chars                                                     \
             count          mean          std    min      25%     50%   
dataset                                                                 
multiturn   4074.0  11028.280560  9076.733204  184.0  3933.25  7623.0   
singleturn  4074.0   1441.454345  1440.518689   24.0   353.25   971.0   

                              n_turns                                        \
                 75%      max   count       mean       std  min   25%   50%   
dataset                                                                       
multiturn   16824.75  44760.0  4074.0  11.217477  7.376802  2.0  10.0  10.0   
singleturn   2139.00  20025.0  4074.0   2.265832  0.937673  2.0   2.0   2.0   

                         
             75%    max  
dataset                  
multiturn   10.0  305.0  
singleturn   2.0   28.0

In [37]:
# save the all_data file
all_data.to_parquet(PROCESSED_DATA_DIR / "all_data.parquet", index=False)


### Generate Embeddings

Multiturn conversations length exceeds the GPT-2 1024-token window so selected a longer context model instead.  Word2Vec doesn't capture positional information which is important in the conversations.  We selected the nomic-ai embedding model because of the length of the data and to ensure we maintain positional context



In [38]:
out_path = PROCESSED_DATA_DIR / "all_data_emb.npy"

# check to make sure the file doesn't exist
# warning that this will take a few hours to 
# create depending on processor available

if not out_path.exists():
    # check type of processor available
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    # create the model and fix the max input length to the models max
    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192

    mask = (all_data["dataset"] == "singleturn").values

    # encode the single turn data with a larger batch size for speed 
    emb_single = model.encode(
        all_data.loc[mask, "conversation"].tolist(),
        batch_size=8,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    # encode the multiturn with smaller batch size
    emb_multi = model.encode(
        all_data.loc[~mask, "conversation"].tolist(),
        batch_size=1,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embeddings = np.empty((len(all_data), emb_single.shape[1]), dtype=emb_single.dtype)
    embeddings[mask] = emb_single
    embeddings[~mask] = emb_multi

    # save embeddings as np.array file
    np.save(out_path, embeddings)



### Review and check the embeddings

In [39]:
# load embeddings
emb = np.load(PROCESSED_DATA_DIR / "all_data_emb.npy") 

In [40]:
# Shape of the embeddings
print(f"Size of the original data: {len(all_data)}")
print(f"Shape of the embeddings: {emb.shape}")

Size of the original data: 8148
Shape of the embeddings: (8148, 768)


### Summary

- Loaded the datasets from both single and multiturn data which was split evenly (4074 rows each)
- Checked each for balance across harm/no harm categories across train/val/test sets (even balance)
- Found that multiturn conversations length will far exceed GPT-2's 1,024 input token limit.  This would cause almost 2/3's of the multiturn row to be truncated at the end which is where the harm generally occurs.
- Instead, we decided to switch our embedding model to the nomic-embed-text-v1.5 to retain 100% of our content because it has a 8,192 input token limit.  
- Generated and verified the embeddings shape and saved to all_data.parquet and all_data_emb.npy 
